# MICE Imputed Datasets 

In [ ]:
import importlib
import feature_engineering as fe_module
import model as model_module
import visualization as viz_module
importlib.reload(fe_module)
importlib.reload(model_module)
importlib.reload(viz_module)
from model import train_best_model, mean_ci
from visualization import plot_feature_importance, plot_confusion_mat, plot_roc, plot_pr_curve, plot_shap_summary
import os
import pandas as pd
import random
import numpy as np

n_variants = 10
fixed_seed = 42   # fixed for reproducibility of the train/test split

progression_types = [
    ("pooled_CN", "CN"),
    ("pooled_MCI_AD", "AD"),
]

# Datasets_MICE has 10 datasets, named variant0 through variant9 they all have the same filestructure. 
files = [
    ("datasets/Datasets_MICE/pooled_CN.csv", "CN"),
    ("datasets/Datasets_MICE/pooled_MCI_AD.csv", "AD"),
]

# params dict: scalar = fixed value, (lo, hi) = int range, (lo, hi, 'log') = log-scale float range
params = {
    'n_estimators':      (100, 1000),
    'max_depth':         (3, 10),
    'learning_rate':     (0.005, 0.3, 'log'),
    'subsample':         (0.2, 1.0),
    'colsample_bytree':  (0.2, 1.0),
    'colsample_bylevel': (0.2, 1.0),
    'colsample_bynode':  (0.2, 1.0),
    'min_child_weight':  (1, 10),
    'gamma':             (0.0, 5.0),
    'reg_alpha':         (1e-4, 10.0, 'log'),
    'reg_lambda':        (1e-4, 10.0, 'log'),
    'max_delta_step':    0,
}

EXPERIMENT  = "experiment_mice_imputed_final"
results_dir = f"experiments/{EXPERIMENT}/grid_results"
charts_dir  = f"experiments/{EXPERIMENT}/charts"
save_dir = f"experiments/{EXPERIMENT}/artifacts"
os.makedirs(results_dir, exist_ok=True)
os.makedirs(charts_dir,  exist_ok=True)
os.makedirs(save_dir, exist_ok=True)

exp_bayesian_results = {}
exp_bayesian_models  = {}    # { key: (model, cols, summary) }


# ── Training ──────────────────────────────────────────────────────────────────
for base_name, prog in progression_types:
    for variant in range(n_variants):
        file_path = f"datasets/Datasets_MICE/variant{variant}/{base_name}.csv"
        df = pd.read_csv(file_path)
        key = f"{base_name}_variant{variant}"
        csv_out = os.path.join(results_dir, f"{key}_cv_scores.csv")
        print(f"\n{'='*60}")
        print(f"=== Optuna search: {key} — {len(df)} samples ===")
        print(f"{'='*60}")

        try:
            model, cols, summary = train_best_model(
                df,
                progression_type=prog,
                params=params,
                csv_path=csv_out,
                save_dir=f"experiments/{EXPERIMENT}",
                n_jobs=10,
                n_trials=1000,
                objective_metric='auc',
                model_base_name=key,
                save_artifacts=True,
                random_state=fixed_seed,
            )
            exp_bayesian_results[key] = pd.read_csv(csv_out)
            exp_bayesian_models[key] = (model, cols, summary)
            np.save(os.path.join(save_dir, f"{key}_y_true.npy"), summary["y_true"])
            np.save(os.path.join(save_dir, f"{key}_y_proba.npy"), summary["y_proba"])
            np.save(os.path.join(save_dir, f"{key}_X_train.npy"), summary["X_train"])

            bm = summary['bootstrap_metrics']
            metrics_dict = {
                'key': key,
                'dataset': base_name,
                'progression_type': prog,
                'variant': variant,
                'base_auc': summary['base_auc'],
                'base_avg_precision': summary['base_avg_precision'],
                'accuracy': bm['accuracy'][0],
                'accuracy_lo': bm['accuracy'][1][0],
                'accuracy_hi': bm['accuracy'][1][1],
                'precision_macro': bm['precision_macro'][0],
                'precision_macro_lo': bm['precision_macro'][1][0],
                'precision_macro_hi': bm['precision_macro'][1][1],
                'recall_macro': bm['recall_macro'][0],
                'recall_macro_lo': bm['recall_macro'][1][0],
                'recall_macro_hi': bm['recall_macro'][1][1],
                'f1_macro': bm['f1_macro'][0],
                'f1_macro_lo': bm['f1_macro'][1][0],
                'f1_macro_hi': bm['f1_macro'][1][1],
                'auc': bm['auc'][0],
                'auc_lo': bm['auc'][1][0],
                'auc_hi': bm['auc'][1][1],
                'ppv': bm['ppv'][0],
                'ppv_lo': bm['ppv'][1][0],
                'ppv_hi': bm['ppv'][1][1],
                'npv': bm['npv'][0],
                'npv_lo': bm['npv'][1][0],
                'npv_hi': bm['npv'][1][1],
            }
            metrics_row = pd.DataFrame([metrics_dict])
            metrics_csv_path = os.path.join(save_dir, 'all_metrics.csv')
            if not os.path.exists(metrics_csv_path):
                metrics_row.to_csv(metrics_csv_path, index=False)
            else:
                metrics_row.to_csv(metrics_csv_path, mode='a', header=False, index=False)

            with open(metrics_csv_path, 'a') as f:
                f.flush()
                os.fsync(f.fileno())

        except Exception as e:
            import traceback
            print(f"Error processing {key}: {e}")
            traceback.print_exc()

# ── Visualizations ────────────────────────────────────────────────────────────
for key, (model, cols, summary) in exp_bayesian_models.items():
    print(f"\n{'='*60}")
    print(f"Visualizations — {key}")
    print(f"{'='*60}")

    # Feature importance
    plot_feature_importance(
        model.feature_importances_,
        cols,
        top_n=20,
        title=f"Top 20 Feature Importances — {key}",
        save_path=os.path.join(charts_dir, f"{key}_feature_importance.png"),
    )

    #Confusion matrix
    plot_confusion_mat(
        summary["y_true"],
        summary["y_pred"],
        title=f"Confusion Matrix — {key}",
        save_path=os.path.join(charts_dir, f"{key}_confusion_matrix.png"),
    )

    # ROC curve
    plot_roc(
        summary["y_true"],
        summary["y_proba"],
        title=f"ROC Curve — {key}",
        save_path=os.path.join(charts_dir, f"{key}_roc_curve.png"),
    )

    # Precision-Recall curve
    plot_pr_curve(
        summary["y_true"],
        summary["y_proba"],
        title=f"Precision-Recall Curve — {key}",
        save_path=os.path.join(charts_dir, f"{key}_pr_curve.png"),
    )

    # SHAP beeswarm (uses stored X_train — no recomputation needed)
    plot_shap_summary(
        model,
        summary["X_train"],
        cols,
        title=f"SHAP Summary — {key}",
        save_path=os.path.join(charts_dir, f"{key}_shap_summary.png"),
        xlim=(-1.0, 1.0),          # standardised x‑axis range
        figsize=(12, 6)            # wider figure
    )
    
### AGGREGATE STATISTICS

print("\n" + "="*60)
print("FINAL MICE AGGREGATE STATISTICS (10 seeds)")
print("="*60)

# Group keys by dataset base name
datasets = {}
for key in exp_bayesian_models.keys():
    base = key.split('_seed')[0]   # e.g., "CN_MCI" or "MCI_AD"
    datasets.setdefault(base, []).append(key)

# We'll collect all aggregate rows in a list for the summary CSV
summary_rows = []

for ds, keys in datasets.items():
    print(f"\n📊 Dataset: {ds}")
    # Collect metrics across seeds for this dataset
    metrics_list = []
    for k in keys:
        _, _, summary = exp_bayesian_models[k]
        bm = summary['bootstrap_metrics']
        metrics_list.append({
            'base_auc': summary['base_auc'],
            'base_avg_precision': summary['base_avg_precision'],
            'accuracy': bm['accuracy'][0],
            'precision_macro': bm['precision_macro'][0],
            'recall_macro': bm['recall_macro'][0],
            'f1_macro': bm['f1_macro'][0],
        })
    df_metrics = pd.DataFrame(metrics_list)

    # Compute and print each metric
    for metric in ['base_auc', 'base_avg_precision', 'accuracy',
                   'precision_macro', 'recall_macro', 'f1_macro']:
        vals = df_metrics[metric].values
        mean_val, lo, hi, std_val = mean_ci(vals)
        print(f"  {metric:20s}: {mean_val:.4f} ± {std_val:.4f}  (95% CI: {lo:.4f} – {hi:.4f})")
        # Store row for CSV
        summary_rows.append({
            'dataset': ds,
            'metric': metric,
            'mean': mean_val,
            'std': std_val,
            'ci_lower': lo,
            'ci_upper': hi,
        })

# Save aggregate stats to CSV
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(save_dir, 'aggregate_stats.csv'), index=False)
print(f"\nAggregate stats saved to: {os.path.join(save_dir, 'aggregate_stats.csv')}")
        



=== Optuna search: pooled_CN_variant0 — 12092 samples ===
Using StratifiedKFold with n_splits=5
Optuna search: 1000 trials, objective=auc, n_jobs=10


Optuna trials:  39%|███▉      | 393/1000 [07:20<10:56,  1.08s/trial]

# Non-Imputed Dataset, 10 seeds of train/test split, full experimental pipeline

In [ ]:
import importlib
import feature_engineering as fe_module
import model as model_module
import visualization as viz_module
importlib.reload(fe_module)
importlib.reload(model_module)
importlib.reload(viz_module)
from model import train_best_model, mean_ci
from visualization import plot_feature_importance, plot_confusion_mat, plot_roc, plot_pr_curve, plot_shap_summary
import os
import pandas as pd
import random
import numpy as np

master_seed = 42
random.seed(master_seed)
seeds = [random.randint(0, 1000) for _ in range(10)]


files = [
    ("datasets/Dataset_v2_1/CN_MCI.csv", "CN"),
    ("datasets/Dataset_v2_1/MCI_AD.csv", "AD"),
]

# params dict: scalar = fixed value, (lo, hi) = int range, (lo, hi, 'log') = log-scale float range
params = {
    'n_estimators':      (100, 1000),
    'max_depth':         (3, 10),
    'learning_rate':     (0.005, 0.3, 'log'),
    'subsample':         (0.2, 1.0),
    'colsample_bytree':  (0.2, 1.0),
    'colsample_bylevel': (0.2, 1.0),
    'colsample_bynode':  (0.2, 1.0),
    'min_child_weight':  (1, 10),
    'gamma':             (0.0, 5.0),
    'reg_alpha':         (1e-4, 10.0, 'log'),
    'reg_lambda':        (1e-4, 10.0, 'log'),
    'max_delta_step':    0,
}

EXPERIMENT  = "experiment_non_imputed_final"
results_dir = f"experiments/{EXPERIMENT}/grid_results"
charts_dir  = f"experiments/{EXPERIMENT}/charts"
save_dir = f"experiments/{EXPERIMENT}/artifacts"
os.makedirs(results_dir, exist_ok=True)
os.makedirs(charts_dir,  exist_ok=True)
os.makedirs(save_dir, exist_ok=True)

exp_bayesian_results = {}
exp_bayesian_models  = {}    # { key: (model, cols, summary) }


# ── Training ──────────────────────────────────────────────────────────────────
for path, prog in files:
    df = pd.read_csv(path)
    base = os.path.splitext(os.path.basename(path))[0]
    for seed in seeds:
        key = f"{base}_seed{seed}"
        csv_out = os.path.join(results_dir, f"{key}_cv_scores.csv")
        print(f"\n{'='*60}")
        print(f"=== Optuna search: {key} — {len(df)} samples ===")
        print(f"{'='*60}")

        try:
            model, cols, summary = train_best_model(
                df,
                progression_type=prog,
                params=params,
                csv_path=csv_out,
                save_dir=f"experiments/{EXPERIMENT}",
                n_jobs=10,
                n_trials=1000,
                objective_metric='auc',
                model_base_name=key,          # use key to avoid overwriting
                save_artifacts=True,
                random_state=seed,            # pass the current seed
            )
            exp_bayesian_results[key] = pd.read_csv(csv_out)
            exp_bayesian_models[key]  = (model, cols, summary)
            np.save(os.path.join(save_dir, f"{key}_y_true.npy"), summary["y_true"])
            np.save(os.path.join(save_dir, f"{key}_y_proba.npy"), summary["y_proba"])
            np.save(os.path.join(save_dir, f"{key}_X_train.npy"), summary["X_train"])
            # Inside the training loop, after saving .npy files
                       # ── Save all metrics to CSV ──────────────────────────────────
            bm = summary['bootstrap_metrics']
            metrics_dict = {
                'key': key,
                'dataset': base,                # e.g. "CN_MCI"
                'progression_type': prog,       # "CN" or "AD"
                'seed': seed,
                'base_auc': summary['base_auc'],
                'base_avg_precision': summary['base_avg_precision'],
                # Accuracy
                'accuracy': bm['accuracy'][0],
                'accuracy_lo': bm['accuracy'][1][0],
                'accuracy_hi': bm['accuracy'][1][1],
                # Precision (macro)
                'precision_macro': bm['precision_macro'][0],
                'precision_macro_lo': bm['precision_macro'][1][0],
                'precision_macro_hi': bm['precision_macro'][1][1],
                # Recall (macro)
                'recall_macro': bm['recall_macro'][0],
                'recall_macro_lo': bm['recall_macro'][1][0],
                'recall_macro_hi': bm['recall_macro'][1][1],
                # F1 (macro)
                'f1_macro': bm['f1_macro'][0],
                'f1_macro_lo': bm['f1_macro'][1][0],
                'f1_macro_hi': bm['f1_macro'][1][1],
                # ROC AUC (same as base_auc, but from bootstrap)
                'auc': bm['auc'][0],
                'auc_lo': bm['auc'][1][0],
                'auc_hi': bm['auc'][1][1],
                # PPV (positive predictive value)
                'ppv': bm['ppv'][0],
                'ppv_lo': bm['ppv'][1][0],
                'ppv_hi': bm['ppv'][1][1],
                # NPV (negative predictive value)
                'npv': bm['npv'][0],
                'npv_lo': bm['npv'][1][0],
                'npv_hi': bm['npv'][1][1],
            }
            metrics_row = pd.DataFrame([metrics_dict])
            metrics_csv_path = os.path.join(save_dir, 'all_metrics.csv')
            if not os.path.exists(metrics_csv_path):
                metrics_row.to_csv(metrics_csv_path, index=False)
            else:
                metrics_row.to_csv(metrics_csv_path, mode='a', header=False, index=False)

            with open(metrics_csv_path, 'a') as f:
                f.flush()
                os.fsync(f.fileno())

        except Exception as e:
            import traceback
            print(f"Error processing {key}: {e}")
            traceback.print_exc()

# ── Visualizations ────────────────────────────────────────────────────────────
for key, (model, cols, summary) in exp_bayesian_models.items():
    print(f"\n{'='*60}")
    print(f"Visualizations — {key}")
    print(f"{'='*60}")

    # Feature importance
    plot_feature_importance(
        model.feature_importances_,
        cols,
        top_n=20,
        title=f"Top 20 Feature Importances — {key}",
        save_path=os.path.join(charts_dir, f"{key}_feature_importance.png"),
    )

    #Confusion matrix
    plot_confusion_mat(
        summary["y_true"],
        summary["y_pred"],
        title=f"Confusion Matrix — {key}",
        save_path=os.path.join(charts_dir, f"{key}_confusion_matrix.png"),
    )

    # ROC curve
    plot_roc(
        summary["y_true"],
        summary["y_proba"],
        title=f"ROC Curve — {key}",
        save_path=os.path.join(charts_dir, f"{key}_roc_curve.png"),
    )

    # Precision-Recall curve
    plot_pr_curve(
        summary["y_true"],
        summary["y_proba"],
        title=f"Precision-Recall Curve — {key}",
        save_path=os.path.join(charts_dir, f"{key}_pr_curve.png"),
    )

    # SHAP beeswarm (uses stored X_train — no recomputation needed)
    plot_shap_summary(
        model,
        summary["X_train"],
        cols,
        title=f"SHAP Summary — {key}",
        save_path=os.path.join(charts_dir, f"{key}_shap_summary.png"),
        xlim=(-1.0, 1.0),          # standardised x‑axis range
        figsize=(12, 6)            # wider figure
    )
    
### AGGREGATE STATISTICS

print("\n" + "="*60)
print("FINAL NON IMPUTED AGGREGATE STATISTICS (10 seeds)")
print("="*60)

# Group keys by dataset base name
datasets = {}
for key in exp_bayesian_models.keys():
    base = key.split('_seed')[0]   # e.g., "CN_MCI" or "MCI_AD"
    datasets.setdefault(base, []).append(key)

# We'll collect all aggregate rows in a list for the summary CSV
summary_rows = []

for ds, keys in datasets.items():
    print(f"\n📊 Dataset: {ds}")
    # Collect metrics across seeds for this dataset
    metrics_list = []
    for k in keys:
        _, _, summary = exp_bayesian_models[k]
        bm = summary['bootstrap_metrics']
        metrics_list.append({
            'base_auc': summary['base_auc'],
            'base_avg_precision': summary['base_avg_precision'],
            'accuracy': bm['accuracy'][0],
            'precision_macro': bm['precision_macro'][0],
            'recall_macro': bm['recall_macro'][0],
            'f1_macro': bm['f1_macro'][0],
        })
    df_metrics = pd.DataFrame(metrics_list)

    # Compute and print each metric
    for metric in ['base_auc', 'base_avg_precision', 'accuracy',
                   'precision_macro', 'recall_macro', 'f1_macro']:
        vals = df_metrics[metric].values
        mean_val, lo, hi, std_val = mean_ci(vals)
        print(f"  {metric:20s}: {mean_val:.4f} ± {std_val:.4f}  (95% CI: {lo:.4f} – {hi:.4f})")
        # Store row for CSV
        summary_rows.append({
            'dataset': ds,
            'metric': metric,
            'mean': mean_val,
            'std': std_val,
            'ci_lower': lo,
            'ci_upper': hi,
        })

# Save aggregate stats to CSV
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(save_dir, 'aggregate_stats.csv'), index=False)
print(f"\nAggregate stats saved to: {os.path.join(save_dir, 'aggregate_stats.csv')}")
        
